## 0. Importing PyTorch and setting up device-agnostic code

In [ ]:
import os
import random
import torch
import torch.nn as nn 

import torchvision
import torchvision.transforms as transforms 
from torch.utils.data import Dataset , DataLoader
s
import requests
import zipfile

from pathlib import Path
from PIL import Image

import matplotlib.pyplot as plt
import numpy as np

torch.__version__

In [ ]:
# Setup device-agnostic code

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device: ", device)

## 1. Get Data

In [ ]:
# setup path to a data folder 

data_path = Path("data/")

image_path = data_path / "pizza_steak_sushi"

# if the image folder doesn't exist then download it and prepare it......

if image_path.is_dir():
    print(f"{image_path} directory already exists... skipping download")

else:
    print(f"{image_path} does not exist, creating one....")
    image_path.mkdir(parents=True, exist_ok=True)

    # Download pizza_steak_sushi data

    with open( data_path / "pizza_steak_sushi.zip", "wb") as f:
        request = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip")

        print("Download pizza , steak and sushi .....")

        f.write(request.content)


    # unzip pizza, steak, sushi data
    with zipfile.ZipFile(data_path / "pizza_steak_sushi.zip", "r") as zip_ref:
        print("Unzipping the pizza, steak and sushi data....")

        zip_ref.extractall(image_path)



## 2. Becoming one with the data (data preparation and data exploration)

In [ ]:
def walk_through_dir(dir_path):
     """
  Walks through dir_path returning its contents.
  Args:
    dir_path (str or pathlib.Path): target directory
  
  Returns:
    A print out of:
      number of subdirectories in dir_path
      number of images (files) in each subdirectory
      name of each subdirectory
  """

     for dirpath, dirnames, filenames in os.walk(dir_path):
          print(f"There are {len(dirnames)} directories and {len(filenames)} images in '{dirpath}. ")

In [ ]:
walk_through_dir(image_path)

In [ ]:
# setup train and testing paths 

train_dir = image_path / "train"

test_dir = image_path / "test"

train_dir, test_dir

## 2.1  Visualizing and Image 
Let's write some code to:
1. Get all of the images paths 
2. Get random image path's using `random.choice.`
3. Get the image class name using ` pathlib.path.parent.stem  `
4. Since we're working with images, let's open the image with Python's PIL
5. We'll then show the image and print metadata.


In [ ]:
# set the seed 

random.seed(42)

# Get all the image paths 
image_path_list  = list(image_path.glob("*/*/*.jpg"))

# pick a random image path 

random_image_path = random.choice(image_path_list)


# Get the image class name from the image path name (the image class is the name of the directory where all image is located.)

image_class = random_image_path.parent.stem

# Open Image 

img = Image.open(random_image_path)


# Print the metadata

print(f"Random Image Path: {random_image_path}")
print(f"Image Class: {image_class}")
print(f"Image height: {img.height}")
print(f"Image width: {img.width}")
img


## 2.2 Visualizing Image using MatPlotlib

In [ ]:
img_as_array = np.asanyarray(img)

plt.figure(figsize=(8, 7))
plt.imshow(img_as_array)
plt.title(f"Image class: {image_class} | Image shape: {img_as_array.shape} -> [height, width, color_channels]")
plt.axis(False);

## 3. Transforming Data
Now what if we wanted to load our image data into PyTorch?

Before we can use our image data with PyTorch we need to:

1. Turn it into tensors (numerical representations of our images).
2. Turn it into a torch.utils.data.Dataset and subsequently a torch.utils.data.DataLoader, we'll call these Dataset and DataLoader for short.


## 3.1 Transforming data with `torchvision.transforms`

In [ ]:
data_transform = transforms.Compose([
    transforms.Resize(size=(64, 64)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor()
])

print(f"Image Size Before transforms: {img.size}")
print(f"Image Size after transforms: {data_transform(img).shape}")
print(f"Image dtype after transforms: {data_transform(img).dtype}")

In [ ]:
def plot_transformed_images(image_paths, transform, n=3, seed=42):

    """Plots a series of random images from image_paths.

    Will open n image paths from image_paths, transform them
    with transform and plot them side by side.

    Args:
        image_paths (list): List of target image paths. 
        transform (PyTorch Transforms): Transforms to apply to images.
        n (int, optional): Number of images to plot. Defaults to 3.
        seed (int, optional): Random seed for the random generator. Defaults to 42.
    """

    random.seed(seed)

    random_image_paths = random.sample(image_paths, k=n)

    for img_path in random_image_paths:
        with Image.open(img_path) as f:
            fig, ax = plt.subplots(1,2)
            ax[0].imshow(f)
            ax[0].set_title(f"Original \nSize: {f.size}")
            ax[0].axis("off")

            # Transform and plot target image 

            transformed_image  = transform(f).permute(1, 2, 0)
            ax[1].imshow(transformed_image)
            ax[1].set_title(f"Transformed \nSize: {transformed_image.shape}")
            ax[1].axis("off")

            fig.suptitle(f"Class: {img_path.parent.stem}", fontsize=16)


plot_transformed_images(image_paths=image_path_list,
                        transform=data_transform,
                        n=3,
                        seed=42)
